In [2]:
import subprocess
import yaml
import os
import sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import tempfile

import threading
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

import warnings
warnings.filterwarnings("ignore")

import pandas as pd

In [ ]:
selected_model = 'resnet18' #'DeiT-Tiny'  # Example model, can be changed
selected_classifier = 'MLPClassifier1' #'logreg'
pretrained=True
pretrain_modality = 'fine_tune' # 'fine_tune','progressive','from_scratch
huggingface = False  # Set to True if using Hugging Face models

output_dir = source_path+f'\\outputs\\online_deep_feature_extraction\\{selected_model}\\{pretrain_modality}'
file_IO.access_or_create_dir(output_dir)
script_name = source_path+"/scripts/fine-tuning.py"
kind = 'patches_224'  # Example kind, can be changed
input_preprocessed=file_IO.load_preprocessed_files(kind, mode='train')
val_filename=file_IO.load_preprocessed_files(kind, mode='val')
use_external_validation = True  # Set to True if using an external validation set

is_progressive = False  # Set to True for progressive training
model_mode = 'truncated'  # 'truncation', 'full', 'truncated'
truncation='remove head'
custom_transform = False  # Set to True for custom transforms
transform_mode = 'resize'  # 'train', 'val', 'test', 'resize'
use_augmentation = True  # Set to True for data augmentation

saved='old-laptop'  # 'new', 'old-laptop', 'new-laptop'
train_df = pd.read_csv(source_path+f'\\outputs\\preprocessed_data\\{input_preprocessed}')
if train_df['file_name'][0].startswith('C'):
    saved = 'new-laptop'

# preparing layer names

In [6]:
transform = u_transforms.get_transform(selected_model, use_patches=True, custom=custom_transform, mode=transform_mode)
backbone = model_utils.get_model(name=selected_model, mode=model_mode, pretrained=pretrained, truncation=truncation)
out=model_utils.test_output(224,transform, backbone,huggingface=False)
in_features = out.shape[1]
print(f"Model {selected_model} output features: {in_features}")

Model resnet18 output features: 512


In [7]:
classificaton_head = model_utils.get_classification_head(selected_classifier,in_features)
model = model_utils.JoinedModels(backbone, classificaton_head)

In [8]:
#https://chatgpt.com/share/687124c2-47a8-8010-a4b0-5124a0bf5ecb
all_param_names = [name for name, _ in model.named_parameters()]
backbone_param_names = [name for name, _ in model.named_parameters() if name.startswith('vision_model.')]
classifier_param_names = [name for name, _ in model.named_parameters() if name.startswith('classifier.')]

In [9]:
print(f"Total parameters: {len(all_param_names)}")
print(f"Backbone parameters: {len(backbone_param_names)}")
print(f"Classifier parameters: {len(classifier_param_names)}")

Total parameters: 64
Backbone parameters: 60
Classifier parameters: 4


In [10]:
print(f"Backbone parameters: {backbone_param_names}")

Backbone parameters: ['vision_model.conv1.weight', 'vision_model.bn1.weight', 'vision_model.bn1.bias', 'vision_model.layer1.0.conv1.weight', 'vision_model.layer1.0.bn1.weight', 'vision_model.layer1.0.bn1.bias', 'vision_model.layer1.0.conv2.weight', 'vision_model.layer1.0.bn2.weight', 'vision_model.layer1.0.bn2.bias', 'vision_model.layer1.1.conv1.weight', 'vision_model.layer1.1.bn1.weight', 'vision_model.layer1.1.bn1.bias', 'vision_model.layer1.1.conv2.weight', 'vision_model.layer1.1.bn2.weight', 'vision_model.layer1.1.bn2.bias', 'vision_model.layer2.0.conv1.weight', 'vision_model.layer2.0.bn1.weight', 'vision_model.layer2.0.bn1.bias', 'vision_model.layer2.0.conv2.weight', 'vision_model.layer2.0.bn2.weight', 'vision_model.layer2.0.bn2.bias', 'vision_model.layer2.0.downsample.0.weight', 'vision_model.layer2.0.downsample.1.weight', 'vision_model.layer2.0.downsample.1.bias', 'vision_model.layer2.1.conv1.weight', 'vision_model.layer2.1.bn1.weight', 'vision_model.layer2.1.bn1.bias', 'vision_

# launch fine-tuning

In [11]:
total_epochs = 110  # Total number of epochs for fine-tuning
weight_decay = 1e-4  # Weight decay for the optimizer 
pretrain_head=5
lr_classific_head= 1e-3  # Learning rate for the classification head
lr_backbone_initial = 1e-5  # Learning rate for the backbone
lr_backbone_final = 1e-8  # Final learning rate for the backbone
#for vit and transformers choose 0.05
steps=training_utils.get_progressive_training_steps(selected_model)

#for progressive fine tuning
optimizer_phases = [pretrain_head]
phase_layers_to_freeze = [backbone_param_names]
phase_lr = [lr_classific_head]
step_phase=8
for i,name in enumerate(steps):
    optimizer_phases.append(optimizer_phases[i]+step_phase) 
    phase_layers=phase_layers_to_freeze[i]
    phase_layers_to_freeze.append([l for l in phase_layers if not(name in l)])
    if i == 0:
        phase_lr.append(lr_backbone_initial)
    else:
        phase_lr.append(phase_lr[i] * 0.1)  # Decrease learning rate for each phase
n = len(steps)+1
#lr_backbone = [lr_backbone_initial + (lr_backbone_final - lr_backbone_initial) * i / (n - 1) for i in range(n)]
phase_lr += [phase_lr[-1] * 0.5] 
phase_optimizer_name='Adam' #'AdamW
optim_config_progressive = {
        'optimizer_phases':optimizer_phases+[total_epochs],  # Example: [10, 10, 80] for 100 epochs
        'phase_layers_to_freeze':phase_layers_to_freeze+[[]],
        'phase_scheduling': ['no_scheduling' for _ in range(len(optimizer_phases)+1)],
        'phase_optimizer':['Adam' for _ in range(len(optimizer_phases)+1)],  # Example: ['AdamW', 'SGD', 'AdamW'] for different phases
        'phase_lr': phase_lr,
        #'phase_optimizer_hyperparams': [{'weight_decay':weight_decay} for _ in range(len(optimizer_phases)+1)],
        'phase_optimizer_hyperparams': [{} for _ in range(len(optimizer_phases)+1)],
        'phase_scheduler_hyperparams': [{} for _ in range(len(optimizer_phases)+1)],
    }

#for full fine tuning
optimizer_phases = [pretrain_head, total_epochs]  # Example: [1, 4, 95] for 100 epochs
optim_config_cosine = {
    'optimizer_phases':[5,15,total_epochs],  # Example: [10, 10, 80] for 100 epochs
    'phase_layers_to_freeze':[backbone_param_names,[],[]],
    'phase_scheduling': ['no_scheduling','Linear','CosineScheduleCustom'],
    'phase_optimizer':['AdamW','AdamW','AdamW'],  # Example: ['AdamW', 'SGD', 'AdamW'] for different phases
    'phase_lr': [1e-3,1e-6,1e-6],
    'phase_optimizer_hyperparams': [{'weight_decay':weight_decay},{'weight_decay':weight_decay},{'weight_decay':weight_decay}],
    'phase_scheduler_hyperparams': [{}, {'warmup_epochs': optimizer_phases[1]}, {'T_max': total_epochs}],
}
optim_config_simple = {
    'optimizer_phases':optimizer_phases,  # Example: [10, 10, 80] for 100 epochs
    'phase_layers_to_freeze':[backbone_param_names,[]],
    'phase_scheduling': ['no_scheduling','no_scheduling'],
    'phase_optimizer':['AdamW','AdamW'],  # Example: ['AdamW', 'SGD', 'AdamW'] for different phases
    'phase_lr': [1e-3,1e-7],
    'phase_optimizer_hyperparams': [{'weight_decay':weight_decay},{'weight_decay':weight_decay}],
    'phase_scheduler_hyperparams': [{}, {}],
}

optimizer_phases = [total_epochs]  # Example: [1, 4, 95] for 100 epochs
optim_config_scratch = {
    'optimizer_phases':optimizer_phases,  # Example: [10, 10, 80] for 100 epochs
    'phase_layers_to_freeze':[[]],
    'phase_scheduling': [],
    'phase_optimizer':['AdamW'],  # Example: ['AdamW', 'SGD', 'AdamW'] for different phases
    'phase_lr': [1e-3],
    'phase_optimizer_hyperparams': [{'weight_decay':weight_decay},{'weight_decay':weight_decay}],
    'phase_scheduler_hyperparams': [{}, {}],
}

In [ ]:
#parameters
args = script_launching.DotDict(
    N_max=282,
    patches=True,
    input_filename=input_preprocessed,
    val_filename=val_filename if use_external_validation else None,
    huggingface=huggingface,
    pooling=False,  # if true in transformer models use pooling, if false only the cls token
    custom_transform=custom_transform,  # custom transform for the dataset
    transform_mode=transform_mode,  # 'train', 'val', 'test'
    save_h5=False,
    selected_model=selected_model,  # googlenet, alexnet
    selected_classifier=selected_classifier,  # 'logreg', 'svm', 'rf', 'gbc', 'mlp', 'dt'
    truncation=truncation,
    running='new-laptop',
    saved=saved,
    model_mode=model_mode,  # 'truncation
    batch_size=64,
    select_cls=False,
    num_workers=4,
    pin_memory=True,
    show_image=True,
    checkpoint_path = output_dir+"\\checkpoint.pt",
    save_path = output_dir,
    total_epochs = total_epochs,
    log_grad_norm = True,
    use_profiler = False,
    run_epochs = 110,
    plot_every = 1,
    patience = 50,
    use_amp = False ,#mixed precision training,
    val_percentage= 1.0 ,#percentage of validation data used for linear evaluation,
    n_splits = 4,
    loss_criterion = 'CrossEntropyLoss',
    optim_config = optim_config_progressive,
    use_augmentation=use_augmentation,  # Set to True for data augmentation
    n_patches=1,
)
file_IO.save_args(args,output_dir)  # Save the arguments to a file

In [ ]:
script_launching.run_experiment_threaded(args,script_name)  # Test a single run first

Starting experiment:
[STDOUT] Output shape:  torch.Size([1, 512])
[STDOUT] tensor([[ 0.1848, -0.4236]])
[STDOUT] Device is:  cuda
[STDOUT] Saving debug images...
[STDOUT] Debug images saved.
[STDOUT] [GPU Memory] Allocated: 0.00 MB | Reserved: 0.00 MB
[STDOUT] Model size: 45.05 MB
[STDOUT] 🔄 Resuming from checkpoint: c:\Users\andre\VsCode\PD related projects\gender_detection\outputs\online_deep_feature_extraction\resnet18\fine_tune\checkpoint.pt
[STDOUT] Setting optimizer for phase 0: Adam with learning rate 0.001
[STDOUT] Freezing ['vision_model.conv1.weight', 'vision_model.bn1.weight', 'vision_model.bn1.bias', 'vision_model.layer1.0.conv1.weight', 'vision_model.layer1.0.bn1.weight', 'vision_model.layer1.0.bn1.bias', 'vision_model.layer1.0.conv2.weight', 'vision_model.layer1.0.bn2.weight', 'vision_model.layer1.0.bn2.bias', 'vision_model.layer1.1.conv1.weight', 'vision_model.layer1.1.bn1.weight', 'vision_model.layer1.1.bn1.bias', 'vision_model.layer1.1.conv2.weight', 'vision_model.laye

# functions

## reload

In [4]:
def reload_modules():
    import importlib
    import utils.data_loading as data_loading
    import utils.visualization as visualization
    import utils.dataframes as dataframes
    import utils.utils_transforms as u_transforms
    import utils.training_utils as training_utils
    import utils.model_utils as model_utils
    import utils.file_IO as file_IO
    import utils.vit_rollout_mod as vit_rollout_mod
    import  utils.script_launching as script_launching
    
    importlib.reload(file_IO)
    importlib.reload(data_loading)
    importlib.reload(visualization)
    importlib.reload(dataframes)
    importlib.reload(u_transforms)
    importlib.reload(model_utils)
    importlib.reload(training_utils)
    importlib.reload(vit_rollout_mod)
    importlib.reload(script_launching)

    return data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, script_launching
data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, script_launching = reload_modules()